In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

In [ ]:
# =========================================================
# SPARK SESSION
# =========================================================

spark = SparkSession.builder \
    .appName("SpotifyClustering") \
    .enableHiveSupport() \
    .getOrCreate()

In [ ]:
# =========================================================
# LOAD DATA FROM HIVE
# =========================================================

# Ganti sesuai nama database dan tabel Hive
# df = spark.sql("""
# SELECT *
# FROM spotify.spotify_tracks
# """)


In [ ]:
# =========================================================
# ALTERNATIVE:
# LOAD DATA FROM HDFS (NON-HIVE)
# =========================================================

# Uncomment jika ingin load langsung dari HDFS
# dan tidak menggunakan Hive

df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("hdfs:///path/to/spotify.csv")




In [ ]:
# =========================================================
# PREPROCESSING
# =========================================================

# -----------------------------------------
# Convert explicit TRUE/FALSE -> 1/0
# -----------------------------------------

df = df.withColumn(
    "explicit_numeric",
    when(col("explicit") == True, 1).otherwise(0)
)


In [ ]:
# -----------------------------------------
# Pilih feature untuk clustering
# -----------------------------------------

feature_columns = [
    "danceability",
    "energy",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "loudness"
]

In [ ]:
# =========================================================
# CREATE CLEAN DATAFRAME
# =========================================================

df_clean = df.select(
    *feature_columns,
    "track_genre"
)

In [ ]:
# =========================================================
# CHECK NULL
# =========================================================

from pyspark.sql.functions import col, sum, when

df_clean.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in feature_columns
]).show()

In [ ]:
# =========================================================
# CAST STRING -> DOUBLE
# =========================================================

for column_name in feature_columns:
    df_clean = df_clean.withColumn(
        column_name,
        col(column_name).cast("double")
    )


In [ ]:
# =========================================================
# DROP NULL
# =========================================================

df_clean = df_clean.dropna(subset=feature_columns)

# Optional debug
df_clean.printSchema()

In [ ]:
# =========================================================
# LIMIT DATA UNTUK TESTING
# =========================================================

df_clean = df_clean.limit(1000)

print(df_clean.count())

In [ ]:
# =========================================================
# VECTOR ASSEMBLER
# =========================================================

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features_raw",
    handleInvalid="skip"
)

df_vector = assembler.transform(df_clean)

In [ ]:
# =========================================================
# STANDARD SCALING
# =========================================================

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(df_vector)

df_scaled = scaler_model.transform(df_vector)

In [ ]:
# =========================================================
# KMEANS
# =========================================================

k = 5

kmeans = KMeans(
    k=k,
    seed=42,
    featuresCol="features",
    predictionCol="cluster"
)

model = kmeans.fit(df_scaled)

df_clustered = model.transform(df_scaled)


In [ ]:
# =========================================================
# EVALUATION
# =========================================================

evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette"
)

silhouette = evaluator.evaluate(df_clustered)

print(f"Silhouette Score: {silhouette}")

In [ ]:
# =========================================================
# SHOW SAMPLE RESULT
# =========================================================

df_clustered.select(
    "track_genre",
    "cluster"
).show(20, truncate=False)


In [ ]:
# =========================================================
# CLUSTER DISTRIBUTION
# =========================================================

df_clustered.groupBy("cluster").count().show()

In [ ]:
# =========================================================
# CLUSTER PROFILE
# =========================================================

cluster_profile = df_clustered.groupBy("cluster").avg(
    *feature_columns
)

cluster_profile.show(truncate=False)

In [ ]:
# =========================================================
# OPTIONAL:
# Dominant genre per cluster
# =========================================================

df_clustered.groupBy(
    "cluster",
    "track_genre"
).count().orderBy(
    "cluster",
    col("count").desc()
).show(100, truncate=False)

In [ ]:
# =========================================================
# VISUALIZATION
# =========================================================

from pyspark.ml.feature import PCA
import matplotlib.pyplot as plt
import pandas as pd

# -----------------------------------------
# PCA 2D
# -----------------------------------------

pca = PCA(
    k=2,
    inputCol="features",
    outputCol="pca_features"
)

pca_model = pca.fit(df_scaled)

df_pca = pca_model.transform(df_clustered)


In [ ]:
# -----------------------------------------
# Extract PCA values
# -----------------------------------------

df_plot = df_pca.select(
    "cluster",
    "pca_features"
).toPandas()

df_plot["pca_x"] = df_plot["pca_features"].apply(lambda x: float(x[0]))
df_plot["pca_y"] = df_plot["pca_features"].apply(lambda x: float(x[1]))

In [ ]:
# -----------------------------------------
# Scatter Plot
# -----------------------------------------

plt.figure(figsize=(10, 8))

for cluster_id in sorted(df_plot["cluster"].unique()):
    cluster_data = df_plot[df_plot["cluster"] == cluster_id]

    plt.scatter(
        cluster_data["pca_x"],
        cluster_data["pca_y"],
        label=f"Cluster {cluster_id}",
        alpha=0.6
    )

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Visualisasi Cluster Spotify")
plt.legend()

plt.show()